<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 40px; margin-top: 0;">
    <div style="flex: 0 0 auto; margin-left: 0; margin-bottom: 0; margin-top: 0;">
        <img src="./pics/UCSD Logo.png" alt="UCSD Logo" style="width: 179px; margin-bottom: 0px; margin-top: 20px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/sdsc-logo.png" alt="SDSC Logo" style="width: 300px; margin-bottom: 0px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/intelimon.png" alt="Intelimon Logo" style="width: 100px; margin-bottom: 0px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/wstc-logo.png" alt="WSTC Logo" style="width: 100px; height: 100px; margin-bottom: 0px;">
    </div>
</div>

<h1 style="text-align: center; font-size: 48px; margin-top: 0;">Shrub Lists Generation</h1>

- **Team Name**: SAMI BAHIG
- **Team Members**:
    - SAMI BAHIG

**Task:**
Review the original and revised scripts, along with the shrub lists produced by each. Provide a general comparison of both workflows and outputs, highlighting the main strengths and limitations you observe in the scripts as well as in the resulting shrub lists.

General Comparison of the Original and Revised IntELiMon Workflows and Shrub Lists
Overview of Both Workflows
Both the original (IntELiMon_1_1_1.R) and revised (IntELiMon_1_1_1_revised.R) scripts share the same foundational pipeline: PTX-to-LAS conversion, Cloth Simulation Filter (CSF) ground classification, DTM generation, height normalization, gap fraction calculation via spherical voxelization, stem detection using the Hough Transform, Leaf Area Density estimation via the MacArthur-Horn equation, and finally understory fuel segmentation. The sole but consequential, differences appear in Step 6, the shrub isolation and classification stage.
Script-Level Comparison
The original script applies a Local Maximum Filter with a 3m window and a minimum height threshold of 1.3m to identify shrub treetops, then runs the Silva (2016) watershed segmentation. Any cluster with a treeID below 200 is automatically classified as a shrub, with no further validation. This approach is simple and fast, but the wide LMF window causes over-segmentation in dense canopy patches, and the 1.3m height floor systematically excludes low-growing shrubs common in Sierra Nevada understories (e.g., species under 1m). Furthermore, the absence of density and geometry checks means that misclassified tree branches, low-hanging foliage, and noise artifacts all end up in the shrub list, inflating counts and degrading label quality.
The revised script addresses these issues through two targeted modifications. First, the LMF window is reduced to 1.5m with a lowered minimum height of 0.25m, enabling detection of smaller and ground-hugging shrubs that the original entirely missed. Second, a post-segmentation validation step filters each detected cluster by three geometric criteria: a minimum of 100 points per cluster (eliminating sparse noise detections), a minimum Z below 0.5m (confirming ground contact and rejecting floating segments such as suspended branches), and a maximum Z range of 2.0m (rejecting tall, vertically elongated objects that are more consistent with tree structure than shrub morphology). Only clusters passing all three criteria receive Classification = 21 and enter the shrub list.
Impact on the Shrub Lists
These algorithmic differences produce meaningfully different CSV outputs. The original shrub list tends to be longer in raw entry count, but a non-trivial proportion of those entries correspond to false positives, branches, dense litter clumps, or segmentation artifacts. The convex hull areas (convhull_area) of these spurious entries tend to be irregular and either unusually small or large, and their Z values may exceed 2m despite being classified in the 0–3m layer. In contrast, the revised shrub list is shorter but geometrically cleaner: entries have lower and more ecologically consistent Z values (typically 0.25–2m), more compact convex hulls, and confirmed ground contact. The trade-off is reduced recall in sparse or low-density areas where genuine shrubs with fewer than 100 points get filtered out.
Strengths and Limitations Summary
The original script's main strength is its permissiveness,  it captures a wider range of candidates and leaves filtering decisions to downstream users. Its main limitation is low precision, making it unreliable as a ground-truth label source without substantial manual cleaning. The revised script's main strength is its higher precision and ecological realism: the validation checks are grounded in the physical properties of shrubs (ground connectivity, limited vertical extent, sufficient point density). Its main limitation is potential under-detection in sparse vegetation or scan-occluded areas where genuine shrubs may fall below the 100-point threshold.
Relevance to Sprint 4
For the downstream shrub detection pipeline in Sprint 4, the revised script's outputs are the preferred label source. In machine learning contexts, label quality directly governs model performance : noisy labels from the original script would propagate systematic errors through training. The revised shrub lists, while smaller, provide a more reliable foundation for model generalization across the full multi-site dataset.

Bonus : A Limitation Common to Both Scripts
Neither script accounts for occlusion bias in 
the shrub inventory. Because TLS scans are captured 
from a single fixed position, shrubs located behind 
tree stems or dense canopy are undersampled or absent
from the point cloud entirely, regardless of which segmentation 
approach is used. This means both shrub lists are inherently 
incomplete representations of true understory fuel loads, and 
any density or count metrics derived from them should be interpreted 
in light of the gap fraction data also produced by the pipeline.